# College FAQ Domain Adapter using PEFT

**Model:** `google/flan-t5-small`  
**Fine-tuning:** LoRA with Hugging Face PEFT  
**Environment:** Google Colab + Google Drive

This notebook builds a small college-FAQ dataset, splits it by FAQ intent to prevent paraphrase leakage, fine-tunes LoRA adapters, and compares the base and adapted models.

## Run the notebook
1. Open in **Google Colab** and select a GPU runtime if available (**Runtime → Change runtime type → T4 GPU**).
2. Choose **Runtime → Run all** and approve the Google Drive access prompt.
3. The notebook creates/updates files under `MyDrive/College_FAQ_PEFT/`.

**Note:** Run All trains the adapter from scratch. Training uses FP32 because an earlier FP16 run produced non-finite loss/logits. The dataset contains paraphrases of 30 underlying FAQ intents—not 630 independent FAQs. Review the answers and official URLs before real-world use.

## 1. Mount Drive and set project paths

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/College_FAQ_PEFT")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "training_results"
ADAPTER_DIR = PROJECT_DIR / "college_faq_lora_adapter"
EVALUATION_DIR = PROJECT_DIR / "evaluation"

for folder in [PROJECT_DIR, DATA_DIR, OUTPUT_DIR, ADAPTER_DIR, EVALUATION_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATASET_PATH = DATA_DIR / "college_faq_dataset.csv"
TRAIN_PATH = DATA_DIR / "train.csv"
VALIDATION_PATH = DATA_DIR / "validation.csv"
TEST_PATH = DATA_DIR / "test.csv"

print("Project folder:", PROJECT_DIR)
print("Dataset and results will be saved to Google Drive.")

## 2. Install the required libraries
The PyTorch version supplied by Colab is retained; only the project libraries are pinned.

In [ ]:
%pip install -q "transformers==4.46.3" "peft==0.13.2" "datasets==3.1.0" "accelerate==1.1.1" sentencepiece scikit-learn safetensors

## 3. Imports and runtime check

In [ ]:
import json
import re
import math
import pandas as pd
import torch
import transformers
import peft
import datasets
import accelerate
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

MODEL_NAME = "google/flan-t5-small"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not detected; training may be slow on CPU.")

## 4. Build the FAQ dataset
Each base FAQ is one intent. Prefix variants are paraphrases used to add wording variety, not additional independent facts.

In [ ]:
# Public institutional sources
ADMISSIONS = "https://iem.edu.in/admissions/"
NOTICES = "https://iem.edu.in/notices/"
CALENDAR = "https://iem.edu.in/academic-calendar/"
ACADEMICS = "https://dean.iem.edu.in/"
CONTACT = "https://iem.edu.in/contact-us/"

# Each entry represents a distinct FAQ intent.
# Answers intentionally direct students to official sources when
# information can change or requires programme-specific confirmation.
faq_facts = [
    ("Admissions", "Where can I find admission information?",
     "Check the official IEM admissions page for current admission information.", ADMISSIONS),
    ("Admissions", "Where can I check eligibility criteria?",
     "Check the official admissions page for the eligibility criteria of your programme.", ADMISSIONS),
    ("Admissions", "How do I find entrance examination information?",
     "Check the official admissions page and the relevant entrance examination portal.", ADMISSIONS),
    ("Admissions", "Where can I confirm the admission process?",
     "Refer to the current admission instructions published on the official IEM website.", ADMISSIONS),
    ("Admissions", "How can I contact the admission office?",
     "Use the current admissions contact details published on the official contact page.", CONTACT),

    ("Examinations", "Where can I find examination notices?",
     "Check the official notices page for the latest examination-related announcements.", NOTICES),
    ("Examinations", "Where can I check the examination timetable?",
     "Look for the latest examination timetable in the official notices and academic portals.", NOTICES),
    ("Examinations", "Where can I find examination form updates?",
     "Check the official notices page for examination form-filling announcements and updates.", NOTICES),
    ("Examinations", "Where can I check result announcements?",
     "Check official college notices and the designated academic portal for result announcements.", NOTICES),
    ("Examinations", "Where can I verify examination rules?",
     "Refer to the official academic regulations and current examination notices.", ACADEMICS),

    ("Fees", "Where can I find the fee structure?",
     "Check the official admissions page for published fee information and verify the applicable programme details.", ADMISSIONS),
    ("Fees", "Where can I find semester fee notices?",
     "Check the official notices page for current semester fee announcements.", NOTICES),
    ("Fees", "How can I confirm a fee payment deadline?",
     "Refer to the latest official fee notice because payment deadlines may change.", NOTICES),
    ("Fees", "Where can I find fee deadline extensions?",
     "Check the official notices page for any published fee deadline extension.", NOTICES),
    ("Fees", "Who should I contact about an administrative fee query?",
     "Use the official contact page to identify the appropriate college office.", CONTACT),

    ("Academic Procedures", "Where can I find the academic calendar?",
     "Consult the official academic calendar for term dates and scheduled academic events.", CALENDAR),
    ("Academic Procedures", "Where can I find academic policies?",
     "Consult the Dean (Academics) portal for academic policies and regulations.", ACADEMICS),
    ("Academic Procedures", "Where can I find syllabus information?",
     "Check the academic portal for published syllabus and curriculum information.", ACADEMICS),
    ("Academic Procedures", "Where can I verify academic regulations?",
     "Refer to the official academic regulations published by the Dean (Academics) office.", ACADEMICS),
    ("Academic Procedures", "Where can I find official academic announcements?",
     "Check the official notices page and the Dean (Academics) portal.", NOTICES),

    ("Internships", "Where can I find information about internships?",
     "Check official college placement information and verified employer announcements.", ADMISSIONS),
    ("Internships", "Where can I verify a placement-related announcement?",
     "Verify the announcement through official college channels or the relevant employer.", CONTACT),
    ("Internships", "Where can I find career-related college information?",
     "Start with official college announcements and contact the appropriate college office for guidance.", CONTACT),
    ("Internships", "How can I check whether an opportunity is genuine?",
     "Verify the opportunity through the employer's official website and the relevant college office.", CONTACT),
    ("Internships", "Who can I contact for college-related career guidance?",
     "Use official college contact channels to identify the appropriate office or coordinator.", CONTACT),

    ("Library and Facilities", "Where can I find information about college facilities?",
     "Check the official college website for published information about facilities.", "https://iem.edu.in/"),
    ("Library and Facilities", "Where can I check library information?",
     "Start with the official college website and confirm current library details with the college.", "https://iem.edu.in/"),
    ("Library and Facilities", "How can I verify library opening hours?",
     "Confirm current library hours through official college information or the library staff.", "https://iem.edu.in/"),
    ("Library and Facilities", "Where can I find information about campus services?",
     "Consult the official college website or contact the relevant college office.", CONTACT),
    ("Library and Facilities", "Where can I find official contact details?",
     "Use the official IEM contact page for published contact information.", CONTACT),
]

# Multiple question phrasings for each intent.
# These are paraphrase variants, not independent factual answers.
question_prefixes = [
    "Please guide me: ",
    "I need help with this question: ",
    "Could you clarify this for me? ",
    "Where should I look to find out: ",
    "Can you tell me: ",
    "I am looking for official information about this: ",
    "What is the correct source for this question: ",
    "Could you help a student with this? ",
    "I have a college-related query: ",
    "Please point me in the right direction: ",
    "How can I verify this information? ",
    "I want to know the official process for this: ",
    "Can you help me find the right information? ",
    "Which official source should I consult for this? ",
    "I need guidance regarding this: ",
    "Where can I confirm the details of this? ",
    "Could you tell me where to check: ",
    "I have a question about college procedures: ",
    "What should I consult for this query? ",
    "Please help me locate the official information: "
]

rows = []

for category, question, answer, source_url in faq_facts:
    # Include the original question and its varied phrasings.
    questions = [question] + [
        prefix + question for prefix in question_prefixes
    ]

    for varied_question in questions:
        rows.append({
            "category": category,
            "question": varied_question,
            "answer": answer,
            "source_url": source_url
        })

df = pd.DataFrame(rows).drop_duplicates(subset=["question"]).reset_index(drop=True)

print("Dataset size:", len(df))
print("Categories:", sorted(df["category"].unique()))
print("\nExamples:")
display(df.head(10))

# Save the CSV
df.to_csv(DATASET_PATH, index=False)
print("\nSaved:", DATASET_PATH)

## 5. Split by intent to prevent data leakage

In [ ]:
# Recover the underlying FAQ wording by removing one known paraphrase prefix.
question_prefixes = [
    "Please guide me: ", "I need help with this question: ",
    "Could you clarify this for me? ", "Where should I look to find out: ",
    "Can you tell me: ", "I am looking for official information about this: ",
    "What is the correct source for this question: ", "Could you help a student with this? ",
    "I have a college-related query: ", "Please point me in the right direction: ",
    "How can I verify this information? ", "I want to know the official process for this: ",
    "Can you help me find the right information? ", "Which official source should I consult for this? ",
    "I need guidance regarding this: ", "Where can I confirm the details of this? ",
    "Could you tell me where to check: ", "I have a question about college procedures: ",
    "What should I consult for this query? ", "Please help me locate the official information: "
]

def get_base_question(question):
    for prefix in question_prefixes:
        if question.startswith(prefix):
            return question[len(prefix):]
    return question

df["intent_id"] = df["question"].apply(get_base_question)
unique_intents = df["intent_id"].drop_duplicates()
train_intents, temp_intents = train_test_split(
    unique_intents, test_size=0.30, random_state=42
)
val_intents, test_intents = train_test_split(
    temp_intents, test_size=5/9, random_state=42
)

train_df = df[df["intent_id"].isin(train_intents)].copy()
val_df = df[df["intent_id"].isin(val_intents)].copy()
test_df = df[df["intent_id"].isin(test_intents)].copy()

for frame, path in [(train_df, TRAIN_PATH), (val_df, VALIDATION_PATH), (test_df, TEST_PATH)]:
    frame.to_csv(path, index=False)

print(f"All rows: {len(df)} | Distinct intents: {df['intent_id'].nunique()}")
for name, frame in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name}: {len(frame)} rows / {frame['intent_id'].nunique()} intents")

assert set(train_df.intent_id).isdisjoint(set(val_df.intent_id))
assert set(train_df.intent_id).isdisjoint(set(test_df.intent_id))
assert set(val_df.intent_id).isdisjoint(set(test_df.intent_id))
assert test_df.intent_id.nunique() >= 5
print("Split files saved in:", DATA_DIR)

## 6. Prepare model inputs

In [ ]:
def prepare_dataset(frame):
    prepared = pd.DataFrame({
        "input_text": (
            "Answer this college FAQ using the information provided: "
            + frame["question"].astype(str)
        ),
        "target_text": frame["answer"].astype(str),
    })
    return Dataset.from_pandas(prepared, preserve_index=False)

train_dataset = prepare_dataset(train_df)
val_dataset = prepare_dataset(val_df)

MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    inputs = tokenizer(
        batch["input_text"], max_length=MAX_INPUT_LENGTH,
        truncation=True, padding="max_length"
    )
    targets = tokenizer(
        text_target=batch["target_text"], max_length=MAX_TARGET_LENGTH,
        truncation=True, padding="max_length"
    )
    inputs["labels"] = [
        [token if token != tokenizer.pad_token_id else -100 for token in seq]
        for seq in targets["input_ids"]
    ]
    return inputs

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
print("Tokenization complete.")

## 7. Load the base model and attach LoRA

In [ ]:
base_for_training = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32
).to(device)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q", "v"],
    bias="none",
)
peft_model = get_peft_model(base_for_training, lora_config)
peft_model.print_trainable_parameters()

trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in peft_model.parameters())
frozen_params = total_params - trainable_params
trainable_percentage = 100 * trainable_params / total_params
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {frozen_params:,}")
print(f"Trainable percentage: {trainable_percentage:.2f}%")

## 8. Configure training and check the initial loss
FP16 and BF16 are disabled for stable FP32 training.

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=False,
    fp16=False,
    bf16=False,
    report_to="none",
    seed=42,
    data_seed=42,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=peft_model, label_pad_token_id=-100
)
trainer = Seq2SeqTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

batch = next(iter(trainer.get_train_dataloader()))
batch = {key: value.to(device) for key, value in batch.items()}
peft_model.eval()
with torch.no_grad():
    sanity_output = peft_model(**batch)

print("Initial batch loss:", float(sanity_output.loss))
assert torch.isfinite(sanity_output.loss).item(), "Non-finite loss; stop and inspect the runtime."
assert torch.isfinite(sanity_output.logits).all().item(), "Non-finite logits; stop and inspect the runtime."
print("Sanity check passed.")

## 9. Fine-tune the adapter

In [ ]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

training_output = trainer.train()
validation_metrics = trainer.evaluate()

print("Fine-tuning completed.")
print("Final training loss:", training_output.training_loss)
print("Validation loss:", validation_metrics.get("eval_loss"))

training_metrics = {
    "model": MODEL_NAME,
    "method": "LoRA with PEFT",
    "epochs": 5,
    "train_steps": trainer.state.global_step,
    "training_loss": float(training_output.training_loss),
    "validation_loss": float(validation_metrics.get("eval_loss", float("nan"))),
    "total_parameters": int(total_params),
    "trainable_parameters": int(trainable_params),
    "frozen_parameters": int(frozen_params),
    "trainable_percentage": float(trainable_percentage),
}
with open(PROJECT_DIR / "training_metrics.json", "w", encoding="utf-8") as f:
    json.dump(training_metrics, f, indent=2)
print("Training metrics saved.")

## 10. Save the adapter and tokenizer to Google Drive

In [ ]:
peft_model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Adapter saved to:", ADAPTER_DIR)
print("Adapter files:", sorted(p.name for p in ADAPTER_DIR.iterdir()))
assert (ADAPTER_DIR / "adapter_config.json").exists()
assert (ADAPTER_DIR / "adapter_model.safetensors").exists()

## 11. Compare the base model and LoRA model on held-out intents

In [ ]:
# Load a fresh base model so the baseline has no LoRA updates.
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32
).to(device).eval()
peft_model.eval()

def generate_answer(model, question):
    prompt = "Answer this college FAQ using the information provided: " + str(question)
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH
    ).to(device)
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=64, num_beams=4)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# One representative question from each of the five held-out intents.
test_unique = test_df.drop_duplicates(subset=["intent_id"]).head(5).copy()
assert len(test_unique) == 5
comparison_rows = []
for _, row in test_unique.iterrows():
    comparison_rows.append({
        "intent_id": row["intent_id"],
        "category": row["category"],
        "question": row["question"],
        "reference_answer": row["answer"],
        "base_answer": generate_answer(baseline_model, row["question"]),
        "lora_answer": generate_answer(peft_model, row["question"]),
        "source_url": row["source_url"],
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_path = EVALUATION_DIR / "comparison_results.csv"
comparison_df.to_csv(comparison_path, index=False)
pd.set_option("display.max_colwidth", 180)
display(comparison_df)
print("Saved comparison:", comparison_path)

## 12. Evaluate ROUGE-L and try five additional questions
ROUGE-L measures lexical overlap only; it does not establish factual accuracy or semantic similarity.

In [ ]:
def tokenize_words(text):
    return re.findall(r"\w+", str(text).lower())

def lcs_length(a, b):
    previous = [0] * (len(b) + 1)
    for token_a in a:
        current = [0] * (len(b) + 1)
        for j, token_b in enumerate(b, start=1):
            if token_a == token_b:
                current[j] = previous[j - 1] + 1
            else:
                current[j] = max(previous[j], current[j - 1])
        previous = current
    return previous[-1]

def rouge_l_f1(reference, prediction):
    ref_tokens, pred_tokens = tokenize_words(reference), tokenize_words(prediction)
    if not ref_tokens or not pred_tokens:
        return 0.0
    common = lcs_length(ref_tokens, pred_tokens)
    precision, recall = common / len(pred_tokens), common / len(ref_tokens)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

metric_rows = []
for _, row in test_unique.iterrows():
    base_answer = comparison_df.loc[comparison_df.intent_id == row.intent_id, "base_answer"].iloc[0]
    lora_answer = comparison_df.loc[comparison_df.intent_id == row.intent_id, "lora_answer"].iloc[0]
    metric_rows.append({
        "intent_id": row["intent_id"],
        "category": row["category"],
        "question": row["question"],
        "reference_answer": row["answer"],
        "base_rougeL_f1": rouge_l_f1(row["answer"], base_answer),
        "lora_rougeL_f1": rouge_l_f1(row["answer"], lora_answer),
    })

metrics_df = pd.DataFrame(metric_rows)
metrics_path = EVALUATION_DIR / "test_evaluation_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
metric_summary = {
    "evaluation_examples": int(len(metrics_df)),
    "distinct_test_intents": int(metrics_df["intent_id"].nunique()),
    "metric": "ROUGE-L F1",
    "mean_base_rougeL_f1": round(float(metrics_df["base_rougeL_f1"].mean()), 4),
    "mean_lora_rougeL_f1": round(float(metrics_df["lora_rougeL_f1"].mean()), 4),
    "note": "Exploratory lexical overlap on five held-out intents; not factual accuracy or semantic similarity.",
}
with open(EVALUATION_DIR / "test_evaluation_summary.json", "w", encoding="utf-8") as f:
    json.dump(metric_summary, f, indent=2)
print(json.dumps(metric_summary, indent=2))
display(metrics_df)

unseen_questions = pd.DataFrame([
    {"category": "Examinations", "question": "Where should I look for updates about a change in my exam schedule?"},
    {"category": "Fees", "question": "What is the best way to check whether my tuition payment has been recorded?"},
    {"category": "Academic Procedures", "question": "Where can I find the dates for the next academic term?"},
    {"category": "Internships", "question": "Where should I look for official information about internship opportunities?"},
    {"category": "Library and Facilities", "question": "How can I find out whether the college library is open on a holiday?"},
])
unseen_questions["base_answer"] = unseen_questions["question"].apply(lambda q: generate_answer(baseline_model, q))
unseen_questions["lora_answer"] = unseen_questions["question"].apply(lambda q: generate_answer(peft_model, q))
unseen_path = EVALUATION_DIR / "unseen_questions_results.csv"
unseen_questions.to_csv(unseen_path, index=False)
display(unseen_questions)
print("Saved unseen-question results:", unseen_path)

## 13. Record runtime resources and evaluation summary

In [ ]:
resource_info = {
    "gpu_available": bool(torch.cuda.is_available()),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA GPU available",
    "gpu_total_memory_GB": round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2) if torch.cuda.is_available() else None,
    "peak_gpu_allocated_memory_GB": round(torch.cuda.max_memory_allocated(0) / (1024 ** 3), 3) if torch.cuda.is_available() else None,
    "peak_gpu_reserved_memory_GB": round(torch.cuda.max_memory_reserved(0) / (1024 ** 3), 3) if torch.cuda.is_available() else None,
    "note": "CUDA peak-memory statistics describe the recorded current runtime and may not represent a prior or restarted run."
}
with open(PROJECT_DIR / "resource_info.json", "w", encoding="utf-8") as f:
    json.dump(resource_info, f, indent=2)

evaluation_summary = {
    **training_metrics,
    **metric_summary,
    "held_out_comparison_file": str(comparison_path.relative_to(PROJECT_DIR)),
    "unseen_question_file": str(unseen_path.relative_to(PROJECT_DIR)),
    "limitations": [
        "The 630 rows are paraphrases of 30 underlying FAQ intents, not 630 independent FAQs.",
        "The test set contains only five distinct intents; results are preliminary.",
        "ROUGE-L measures lexical overlap, not factual accuracy or semantic similarity.",
        "Generated answers may be generic; verify all answers and URLs against official college sources."
    ]
}
with open(PROJECT_DIR / "evaluation_summary.json", "w", encoding="utf-8") as f:
    json.dump(evaluation_summary, f, indent=2)
print("Resource information saved.")
print(json.dumps(evaluation_summary, indent=2))

## 14. Create the project README

In [ ]:
readme_content = f"""# College FAQ Domain Adapter using PEFT

## Project overview
Fine-tunes `google/flan-t5-small` with LoRA (PEFT) to generate college FAQ answers. This is a mini-project demonstrating parameter-efficient fine-tuning, not a production-ready student support system.

## Technology stack
- Python and Google Colab
- PyTorch, Hugging Face Transformers and Datasets
- PEFT (LoRA), Pandas, scikit-learn

## Dataset
- **Rows:** {{len(df)}}
- **Underlying FAQ intents:** {{df['intent_id'].nunique()}}
- **Categories:** Admissions, Examinations, Fees, Academic Procedures, Internships, Library and Facilities
- **Split:** train {{len(train_df)}} rows / {{train_df['intent_id'].nunique()}} intents; validation {{len(val_df)}} / {{val_df['intent_id'].nunique()}}; test {{len(test_df)}} / {{test_df['intent_id'].nunique()}}

Each intent has the original question plus prefixed paraphrases. These are not independent facts. Splitting by intent prevents variants of the same FAQ from leaking between splits.

## Model and training
- Base model: `google/flan-t5-small`
- Method: LoRA with PEFT; target modules `q` and `v`; rank 8; alpha 16
- Epochs: 5; training steps: {{trainer.state.global_step}}
- Precision: FP32 (FP16/BF16 disabled)
- Final recorded training loss: {{training_output.training_loss:.4f}}
- Final validation loss: {{validation_metrics.get('eval_loss', float('nan')):.4f}}

## Parameter efficiency
| Measurement | Value |
|---|---:|
| Total parameters | {{total_params:,}} |
| Trainable parameters | {{trainable_params:,}} |
| Frozen parameters | {{frozen_params:,}} |
| Trainable percentage | {{trainable_percentage:.2f}}% |

## Evaluation
ROUGE-L F1 is an exploratory word-sequence overlap metric on one example from each of five held-out intents.

| Model | Mean ROUGE-L F1 |
|---|---:|
| Base | {{metric_summary['mean_base_rougeL_f1']:.4f}} |
| LoRA | {{metric_summary['mean_lora_rougeL_f1']:.4f}} |

This small test is not a factual-accuracy or semantic-similarity evaluation. See `evaluation/comparison_results.csv`, `evaluation/test_evaluation_metrics.csv`, and `evaluation/unseen_questions_results.csv` for outputs.

## Files
- `data/`: generated FAQ dataset and intent-separated train/validation/test CSVs
- `training_results/`: Trainer checkpoints and logs
- `college_faq_lora_adapter/`: saved LoRA adapter and tokenizer
- `evaluation/`: baseline comparison, unseen-question demo and ROUGE-L results
- `training_metrics.json`, `evaluation_summary.json`, `resource_info.json`: recorded metrics

## Reproduce
1. Open this notebook in Google Colab.
2. Select a GPU runtime if available.
3. Choose **Runtime → Run all** and authorize Google Drive.
4. Wait for training, evaluation and artifact checks to finish.

## Limitations and responsible use
- The data is synthetic and represents only 30 underlying FAQ intents.
- Some answers are generic, and the test set is very small.
- ROUGE-L is lexical overlap, not factual correctness.
- Verify every FAQ answer and URL against current official IEM sources before deployment. Do not use the model as the sole source for deadlines, fees, eligibility or other changing information.
"""
readme_path = PROJECT_DIR / "README.md"
readme_path.write_text(readme_content, encoding="utf-8")
print("README saved:", readme_path)

## 15. Final artifact check

In [ ]:
required_paths = [
    DATASET_PATH, TRAIN_PATH, VALIDATION_PATH, TEST_PATH,
    ADAPTER_DIR / "adapter_config.json", ADAPTER_DIR / "adapter_model.safetensors",
    PROJECT_DIR / "README.md", PROJECT_DIR / "training_metrics.json",
    PROJECT_DIR / "evaluation_summary.json", PROJECT_DIR / "resource_info.json",
    EVALUATION_DIR / "comparison_results.csv",
    EVALUATION_DIR / "test_evaluation_metrics.csv",
    EVALUATION_DIR / "test_evaluation_summary.json",
    EVALUATION_DIR / "unseen_questions_results.csv",
]
print("FINAL FILE CHECK")
for path in required_paths:
    print(f"{'OK' if path.exists() else 'MISSING'}  {path.relative_to(PROJECT_DIR)}")
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, "Missing expected artifacts: " + ", ".join(missing)
print("\nAll expected submission artifacts are present.")
print("Google Drive project folder:", PROJECT_DIR)

## Results and limitations

- The adapter is trained with LoRA; the base model parameters remain frozen.
- Compare `base_answer` and `lora_answer` in the evaluation CSVs rather than assuming every generated answer is correct.
- The dataset is small and synthetic, and the quantitative test uses only five distinct intents. Treat the results as a demonstration of the PEFT workflow, not a robust accuracy claim.